# Jev as Detect: two gates, codebook v1.1

Jev (TypeSafe, System One) through OpenRouter's Decisions API. One call per episode asks:

| question | type | returns | role |
|---|---|---|---|
| `is_contestation` | noul | P(dispute) | gate 1 → Verify → Hold / Revise / Surface |
| `explanation_request` | noul | P(request) | gate 2 → Re-present |
| `presence`, `target`, `interaction_act`, `grounds`, `expected_response` | choice | value + probabilities | **features only**, never labels (DATA_RULES §7) |

Changes from run 1 (codebook v1.0, 29 Oct 2026): the codebook rules sit in the questions'
`instructions`/`criteria`, not in `state` (run 1 showed Jev ignores long state text); `state` is
the unit-rule window (target ± 2 lines, same-speaker continuation only), with the full ±4 window
as an ablation; three repeats on gold; thresholds set by the rule in `data/DATA_RULES.md`, not by
eye; nothing is written to `labels.db`.

Order of work, as the rules require: gold first (tiers A+B, `in_guide` excluded), thresholds
fixed and recorded, then the 250 once.

In [ ]:
import json
import os
import time
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.metrics import cohen_kappa_score, precision_recall_fscore_support, roc_auc_score

HERE = Path.cwd()
ROOT = next(p for p in [HERE, *HERE.parents] if (p / "data" / "gold_standard.csv").exists())
DATA, OUT = ROOT / "data", ROOT / "benchmark" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
CACHE = OUT / "jev_detect_cache.csv"

JEV_URL = "https://openrouter.ai/api/alpha/decisions"
JEV_MODEL = "typesafe/jev-1.14"
EXPECTED_SNAPSHOT = "typesafe/jev-1.14-20261014"   # DATA_RULES §9: a different snapshot is a new run
N_REPEATS = 3                                       # DATA_RULES §8
WINDOW = "unit"                                     # "unit" (default) | "full" (ablation)
PRECISION_FLOOR = 0.75                              # DATA_RULES, operating-point rule
CODEBOOK_VERSION = "1.1"
RUN_DATE = date.today().isoformat()

TAXONOMY = json.loads((DATA / "taxonomy_v1.1.json").read_text())
assert TAXONOMY["version"] == CODEBOOK_VERSION
DIMS = {d["name"]: d["values"] for d in TAXONOMY["dimensions"] if isinstance(d["values"], list)}
CHOICE_DIMS = ["presence", "target", "interaction_act", "grounds", "expected_response"]

gold = pd.read_csv(DATA / "gold_standard.csv", keep_default_na=False)
todo = pd.read_csv(DATA / "episodes_to_label.csv", keep_default_na=False)
tiers = pd.read_csv(DATA / "gold_tiers.csv", keep_default_na=False)
gold = gold.merge(tiers, on="episode_id")
gold["in_guide"] = gold["in_guide"].astype(str).str.lower().eq("true")
EVAL = gold[~gold["in_guide"]]                      # DATA_RULES §5
print(f"gold {len(gold)} (tier A {int((gold.tier == 'A').sum())}, B {int((gold.tier == 'B').sum())}; "
      f"{int(gold.in_guide.sum())} in_guide excluded → {len(EVAL)} evaluated)  |  to label {len(todo)}")

In [ ]:
def load_key() -> str:
    key = os.environ.get("OPENROUTER_API_KEY", "")
    if not key and (ROOT / ".env").exists():
        for line in (ROOT / ".env").read_text().splitlines():
            if line.startswith("OPENROUTER_API_KEY="):
                key = line.split("=", 1)[1].strip().strip("\"'")
    if not key:
        raise RuntimeError("set OPENROUTER_API_KEY in the environment or in .env at the repo root")
    return key


API_KEY = load_key()
print("key loaded:", API_KEY[:8] + "…")

## Questions

The codebook goes into the two gate questions. Option descriptions for the five feature questions
are the ones the human coders see in the app's Guide tab.

In [ ]:
RULES = (f"{TAXONOMY['definition']} {TAXONOMY['core_test']} Unit: {TAXONOMY['unit_rule']} "
         "Rules: " + " ".join(TAXONOMY["boundary_rules"]) + " Also not contestation: " + " ".join(TAXONOMY["also_no"]))
ER = TAXONOMY["explanation_request"]
REQUEST_RULES = f"{ER['definition']} Test: {ER['test']} Not a request: " + " ".join(ER["not_a_request"]) + f" {ER['relation_to_contestation']}"

DESCRIPTIONS = {
    "presence": {
        "explicit": "The challenge is stated directly and unambiguously.",
        "implicit": "The challenge is conveyed through tone, phrasing, or implication without being stated outright.",
        "ambiguous": "You cannot tell whether it is a genuine challenge or mere incomprehension / uncertainty.",
    },
    "target": {
        "prediction": "The model's output / classification for this instance.",
        "reasoning": "How the model arrived at its output.",
        "evidence": "The features or data the explanation uses to justify the output.",
        "input data": "The quality, completeness, or correctness of the underlying data.",
        "explanation representation": "The chart or visual itself, when the claim is that it prevents correct interpretation.",
        "system competence": "The AI's general ability or authority to judge in this domain.",
    },
    "interaction_act": {
        "request for justification": "Asks why: 'Why does this feature matter here?'",
        "direct challenge": "Outright assertion that the output or reasoning is wrong.",
        "correction": "Offers the right answer or fixes an error.",
        "alternative proposal": "Suggests a different model, feature, or approach.",
        "counterexample": "Gives a concrete case that contradicts the system's claim.",
        "rejection": "Dismisses the output or explanation without offering an alternative.",
        "request to acknowledge uncertainty": "Asks the system to be less confident.",
    },
    "grounds": {
        "domain knowledge": "Expertise about mushrooms / biology.",
        "technical knowledge": "Expertise about ML, statistics, or XAI methods.",
        "prior experience": "Personal experience collecting or identifying mushrooms.",
        "internal inconsistency": "The explanation contradicts itself.",
        "missing evidence": "Important features or information are absent from the explanation.",
        "data-quality concern": "The underlying data is wrong, incomplete, or unreliable.",
        "causal implausibility": "The feature-outcome relationship doesn't make biological sense.",
        "presentation ambiguity": "The visual or wording is ambiguous in a way that affects interpretation.",
    },
    "expected_response": {
        "justify": "Explain why the model made this decision.",
        "revise": "Change or update the model / output.",
        "incorporate correction": "Accept and act on the participant's correction.",
        "provide additional information": "Supply missing context or data.",
        "acknowledge uncertainty": "Admit the model is not confident here.",
        "expose conflicting evidence": "Show the evidence that conflicts with the participant's claim.",
        "defer to human expertise": "Yield to the participant's domain authority.",
    },
}
for dim in CHOICE_DIMS:
    assert set(DESCRIPTIONS[dim]) == set(DIMS[dim]), dim
INSTRUCTIONS = {
    "presence": "If the target line is contestation, how explicit is the challenge?",
    "target": "If the target line is contestation, what is being challenged?",
    "interaction_act": "If the target line is contestation, what move is the participant making?",
    "grounds": "If the target line is contestation, what is the challenge based on?",
    "expected_response": "If the target line is contestation, what would resolve the challenge?",
}
key_of = lambda v: v.replace("-", " ").replace(" ", "_")
VALUE_OF = {dim: {key_of(v): v for v in DESCRIPTIONS[dim]} for dim in CHOICE_DIMS}

QUESTIONS = {
    "is_contestation": {
        "type": "noul",
        "instructions": RULES,
        "criteria": {
            "true": "In the target line or its same-speaker continuation the participant disputes what the "
                    "system asserts or does (its prediction, reasoning, evidence, data, or competence, or a "
                    "display that blocks or distorts the correct reading), calling for justification, "
                    "correction, reconsideration, or acknowledgment of uncertainty.",
            "false": "No dispute of content: incomprehension, a request for a different form or level of "
                     "explanation, a stated preference about looks, personal uncertainty, idle speculation, "
                     "reading or interpreting the slide, agreement, general affect, or the interviewer's "
                     "task question.",
        },
    },
    "explanation_request": {
        "type": "noul",
        "instructions": REQUEST_RULES,
        "criteria": {
            "true": "A different explanation would have helped the participant at this line: another format, "
                    "a plainer or less mathematical level, an added legend, axis label, unit, term definition, "
                    "or datum; includes saying they cannot read or understand it or asking what something means.",
            "false": "The participant reads, interprets, agrees, or disputes the content on its merits, or "
                     "comments on themselves or their feelings, with no sign that the explanation's form or "
                     "level failed them.",
        },
    },
    **{dim: {"type": "choice", "instructions": INSTRUCTIONS[dim],
             "criteria": {key_of(v): d for v, d in DESCRIPTIONS[dim].items()}} for dim in CHOICE_DIMS},
}


def unit_window(context: str, target: str) -> str:
    """Target line ± 2 lines; after the target only the same speaker's lines until an INT: turn."""
    lines = [l.strip() for l in context.split("||") if l.strip()]
    t = target.strip()
    idx = next((i for i, l in enumerate(lines) if t and (t in l or l.endswith(t[-60:]))), None)
    if idx is None:
        return context
    after = []
    for l in lines[idx + 1: idx + 3]:
        if l.startswith("INT"):
            break
        after.append(l)
    return " || ".join(lines[max(0, idx - 2): idx] + [lines[idx]] + after)


def build_state(ep: pd.Series, window: str = WINDOW) -> dict:
    ctx_pl = unit_window(ep["context_pl"], ep["text_pl"]) if window == "unit" else ep["context_pl"]
    ctx_en = unit_window(ep["context_en"], ep["text_en"]) if window == "unit" else ep["context_en"]
    return {
        "task": "Think-aloud interview about an AI mushroom-edibility classifier and its explanations. "
                "Judge the TARGET LINE. Polish is authoritative; English is a translation. "
                "Lines are tagged INT: (interviewer) and P: (participant).",
        "episode": {
            "speaker_of_target_line": ep["speaker"],
            "participant_group": ep["participant_group"],
            "explanation_format": ep["explanation_format"],
            "context_pl": ctx_pl, "target_line_pl": ep["text_pl"],
            "context_en": ctx_en, "target_line_en": ep["text_en"],
        },
    }


_s = build_state(gold.iloc[0])
print("state bytes (unit window):", len(json.dumps(_s, ensure_ascii=False).encode()),
      "| questions bytes:", len(json.dumps(QUESTIONS).encode()))
print(_s["episode"]["context_en"][:400])

## Jev call and cache

In [ ]:
def jev_decide(state: dict, questions: dict, tries: int = 5) -> dict:
    body = {"model": JEV_MODEL, "state": state, "questions": questions}
    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json",
               "X-Title": "XAI contestation detect"}
    for attempt in range(tries):
        try:
            r = requests.post(JEV_URL, headers=headers, json=body, timeout=60)
        except requests.RequestException:
            if attempt == tries - 1:
                raise
            time.sleep(2 ** (attempt + 1)); continue
        if r.status_code == 429 or r.status_code >= 500:
            if attempt == tries - 1:
                r.raise_for_status()
            time.sleep(2 ** (attempt + 1)); continue
        if r.status_code != 200:
            raise RuntimeError(f"Jev {r.status_code}: {r.text[:500]}")
        return r.json()


def jev_score(ep: pd.Series, window: str) -> dict:
    res = jev_decide(build_state(ep, window), QUESTIONS)
    ans = res["answers"]
    model = res.get("model", JEV_MODEL)
    if model != EXPECTED_SNAPSHOT:
        print(f"\nWARNING snapshot {model} != {EXPECTED_SNAPSHOT}")
    return {"p_contestation": round(float(ans["is_contestation"]["noul"]), 4),
            "p_request": round(float(ans["explanation_request"]["noul"]), 4),
            **{f"f_{d}": VALUE_OF[d][ans[d]["choice"]] for d in CHOICE_DIMS},
            "answers": json.dumps(ans, ensure_ascii=False), "model": model,
            "cost": res.get("usage", {}).get("cost", "")}


CACHE_COLS = ["episode_id", "split", "repeat", "window", "codebook", "run_date", "p_contestation", "p_request",
              *[f"f_{d}" for d in CHOICE_DIMS], "answers", "model", "cost"]


def load_cache() -> pd.DataFrame:
    return pd.read_csv(CACHE, keep_default_na=False) if CACHE.exists() else pd.DataFrame(columns=CACHE_COLS)


def run(episodes: pd.DataFrame, split: str, repeat: int = 0, window: str = WINDOW) -> pd.DataFrame:
    cache = load_cache()
    sel = (cache["split"] == split) & (cache["repeat"].astype(str) == str(repeat)) & (cache["window"] == window)
    done = set(cache.loc[sel, "episode_id"])
    for i, (_, ep) in enumerate(episodes.iterrows()):
        if ep["episode_id"] in done:
            continue
        row = {"episode_id": ep["episode_id"], "split": split, "repeat": repeat, "window": window,
               "codebook": CODEBOOK_VERSION, "run_date": RUN_DATE, **jev_score(ep, window)}
        pd.DataFrame([row]).reindex(columns=CACHE_COLS, fill_value="").to_csv(
            CACHE, mode="a", header=not CACHE.exists(), index=False)
        print(f"\r{split} r{repeat} {window}: {i + 1}/{len(episodes)}", end="")
    print()
    cache = load_cache()
    sel = (cache["split"] == split) & (cache["repeat"].astype(str) == str(repeat)) & (cache["window"] == window)
    res = cache[sel].drop_duplicates("episode_id", keep="last").copy()
    res["p_contestation"] = pd.to_numeric(res["p_contestation"]); res["p_request"] = pd.to_numeric(res["p_request"])
    print(f"{split} r{repeat} {window}: {len(res)} episodes, cost ${pd.to_numeric(res['cost'], errors='coerce').sum():.4f}, "
          f"snapshots {sorted(res['model'].unique())}")
    return res

## 1. Gold, three repeats (tier A+B, `in_guide` excluded)

Per-episode mean P over repeats is the score. Variance between repeats is reported, not hidden.

In [ ]:
reps = [run(gold, "gold", repeat=r) for r in range(N_REPEATS)]
G = EVAL[["episode_id", "tier", "is_contestation", "presence"]].copy()
P = pd.concat([r.set_index("episode_id")["p_contestation"].rename(f"r{i}") for i, r in enumerate(reps)], axis=1)
R = pd.concat([r.set_index("episode_id")["p_request"].rename(f"r{i}") for i, r in enumerate(reps)], axis=1)
G["p_c"], G["p_c_sd"] = P.reindex(G.episode_id).mean(axis=1).values, P.reindex(G.episode_id).std(axis=1).values
G["p_r"], G["p_r_sd"] = R.reindex(G.episode_id).mean(axis=1).values, R.reindex(G.episode_id).std(axis=1).values
y = (G["is_contestation"] == "yes").astype(int)

print(f"n={len(G)}  AUC(mean P)={roc_auc_score(y, G.p_c):.3f}  "
      f"per-repeat AUC={[round(roc_auc_score(y, P.reindex(G.episode_id)[c]), 3) for c in P.columns]}")
print(f"run-to-run sd of P(contestation): median {G.p_c_sd.median():.3f}, max {G.p_c_sd.max():.3f}")
print("P(contestation) by gold:"); print(G.groupby("is_contestation").p_c.describe()[["mean", "25%", "50%", "75%"]].round(2))

## 2. Threshold by rule (DATA_RULES: lowest threshold with precision ≥ 0.75 on gold)

`explanation_request` has no gold labels yet (new in v1.1), so its threshold cannot be set by the
same rule. Until v1.1 human labels exist it uses 0.5 and is marked provisional.

In [ ]:
def sweep(y: pd.Series, p: pd.Series) -> pd.DataFrame:
    rows = []
    for t in np.round(np.arange(0.05, 0.96, 0.01), 2):
        pred = (p >= t).astype(int)
        if pred.sum() == 0:
            continue
        pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average=None, labels=[0, 1], zero_division=0)
        rows.append({"t": t, "precision": pr[1], "recall": rc[1], "f1_yes": f1[1], "macro_f1": f1.mean(),
                     "kappa": cohen_kappa_score(y, pred), "pred_yes": int(pred.sum())})
    return pd.DataFrame(rows).set_index("t")


S = sweep(y, G.p_c)
ok = S[S.precision >= PRECISION_FLOOR]
if len(ok):
    T_C = float(ok.index.min())
    note_c = f"lowest t with precision ≥ {PRECISION_FLOOR}"
else:
    T_C = float(S.macro_f1.idxmax())
    note_c = f"NO threshold reaches precision {PRECISION_FLOOR}; fell back to max macro-F1"
T_R, note_r = 0.5, "provisional: no v1.1 gold for explanation_request yet"
print(f"T_C = {T_C} ({note_c})\n")
print(S.loc[[t for t in [0.2, 0.3, 0.4, 0.5, 0.6, T_C] if t in S.index]].round(3).drop_duplicates())

# per-tier view: A alone only next to A+B (DATA_RULES §4)
for name, sub in [("A+B", G), ("A", G[G.tier == "A"])]:
    yy, pp = (sub.is_contestation == "yes").astype(int), (sub.p_c >= T_C).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(yy, pp, average=None, labels=[0, 1], zero_division=0)
    print(f"tier {name:3s} n={len(sub):2d} @ {T_C}: precision={pr[1]:.2f} recall={rc[1]:.2f} macroF1={f1.mean():.2f} "
          f"kappa={cohen_kappa_score(yy, pp):.2f}")
per_rep = [cohen_kappa_score(y, (P.reindex(G.episode_id)[c] >= T_C).astype(int)) for c in P.columns]
print(f"kappa per repeat @ {T_C}: {np.round(per_rep, 3)}  mean {np.mean(per_rep):.3f} ± {np.std(per_rep):.3f}")

decision = {"run_date": RUN_DATE, "codebook": CODEBOOK_VERSION, "model": sorted(set(reps[0].model)),
            "window": WINDOW, "n_repeats": N_REPEATS, "n_gold_evaluated": int(len(G)),
            "threshold_contestation": T_C, "rule_contestation": note_c,
            "threshold_request": T_R, "rule_request": note_r}
(OUT / "detect_thresholds.json").write_text(json.dumps(decision, indent=1))
line = (f"- {RUN_DATE}: Detect thresholds fixed on gold (n={len(G)}, {N_REPEATS} repeats, window={WINDOW}, "
        f"codebook v{CODEBOOK_VERSION}): contestation {T_C} ({note_c}); request {T_R} ({note_r}).")
rules = (DATA / "DATA_RULES.md").read_text()
if line not in rules:
    (DATA / "DATA_RULES.md").write_text(rules.rstrip("\n") + "\n" + line + "\n")
    print("recorded in data/DATA_RULES.md")
else:
    print("already recorded in data/DATA_RULES.md")

In [ ]:
# Disagreements at T_C, for the re-adjudication queue (reading, not labels)
G["jev"] = np.where(G.p_c >= T_C, "yes", "no")
dis = G[G.jev != G.is_contestation].merge(gold[["episode_id", "text_en"]], on="episode_id")
print(f"{len(dis)} disagreements at {T_C}: {int((dis.is_contestation == 'yes').sum())} misses, "
      f"{int((dis.is_contestation == 'no').sum())} false alarms; {int((dis.tier == 'B').sum())} of them already tier B")
dis[["episode_id", "tier", "is_contestation", "presence", "p_c", "p_c_sd", "p_r", "text_en"]].sort_values("p_c")

## 3. Window ablation: full ±4 context, one repeat

In [ ]:
full = run(gold, "gold", repeat=0, window="full").set_index("episode_id").reindex(G.episode_id)
for name, p in [("unit (mean of 3)", G.p_c.values), ("full (1 run)", full.p_contestation.values)]:
    pred = (p >= T_C).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average=None, labels=[0, 1], zero_division=0)
    print(f"{name:18s} AUC={roc_auc_score(y, p):.3f} @ {T_C}: precision={pr[1]:.2f} recall={rc[1]:.2f} "
          f"macroF1={f1.mean():.2f} kappa={cohen_kappa_score(y, pred):.2f}")

## 4. Explanation request on gold (descriptive only)

No v1.1 gold labels exist yet, so this is what Jev *says*, not a score. The overlap with the
contestation gate is the thing to look at: requests should be common where gold says
`target = explanation representation` and among gold negatives that are incomprehension.

In [ ]:
G["request"] = np.where(G.p_r >= T_R, "yes", "no")
print(pd.crosstab(G.is_contestation, G.request, rownames=["gold contestation"], colnames=["jev request"]))
tg = G.merge(gold[["episode_id", "target"]], on="episode_id")
print("\nP(request) by gold target:"); print(tg.groupby("target").p_r.agg(["mean", "count"]).round(2))

## 5. The 250: silver labels, one run

Thresholds come from section 2 and are not touched here. Output `data/silver_250.csv` carries the
two gate labels and probabilities only; the five feature answers stay in the cache
(DATA_RULES §1, §7).

In [ ]:
silver = run(todo, "to_label", repeat=0)
silver = silver.merge(todo[["episode_id", "transcript_id", "participant_group", "explanation_format"]], on="episode_id")
silver["is_contestation"] = np.where(silver.p_contestation >= T_C, "yes", "no")
silver["explanation_request"] = np.where(silver.p_request >= T_R, "yes", "no")
silver["threshold_contestation"], silver["threshold_request"] = T_C, T_R
silver["codebook"], silver["source"] = CODEBOOK_VERSION, "jev zero-shot (silver)"
cols = ["episode_id", "transcript_id", "participant_group", "explanation_format", "is_contestation",
        "explanation_request", "p_contestation", "p_request", "threshold_contestation", "threshold_request",
        "model", "codebook", "run_date", "window", "source"]
silver[cols].to_csv(DATA / "silver_250.csv", index=False)
print(pd.crosstab(silver.is_contestation, silver.explanation_request, rownames=["contestation"], colnames=["request"]))
silver.pivot_table(index="participant_group", columns="explanation_format", values="is_contestation",
                   aggfunc=lambda s: (s == "yes").sum(), fill_value=0)